# Notebook 3: Feature Engineering

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

print(" Starting Feature Engineering...")
df = pd.read_csv("../data/nifty_merged_5min.csv", index_col='timestamp', parse_dates=True)

# 1. EMA Indicators (Task 2.1)
df['ema_5'] = df['close'].ewm(span=5, adjust=False).mean()
df['ema_15'] = df['close'].ewm(span=15, adjust=False).mean()

# 2. Option Greeks (Task 2.2 - Simplified Black-Scholes)
# Note: Full Black-Scholes requires strict T (time to expiry). We simulate T here.
def calculate_delta(S, K, r, sigma, T, option_type='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    if option_type == 'call':
        return norm.cdf(d1)
    else:
        return norm.cdf(d1) - 1

df['T'] = 7/365 # Assume constant 7 days to expiry for simplicity
df['call_delta'] = calculate_delta(df['close'], df['atm_strike'], 0.065, df['call_iv']/100, df['T'], 'call')
df['put_delta'] = calculate_delta(df['close'], df['atm_strike'], 0.065, df['put_iv']/100, df['T'], 'put')

# 3. Derived Features (Task 2.3)
df['average_iv'] = (df['call_iv'] + df['put_iv']) / 2
df['pcr_oi'] = df['put_oi'] / df['call_oi']
df['futures_basis'] = (df['close_fut'] - df['close']) / df['close']
df['returns'] = df['close'].pct_change()
df['delta_neutral_ratio'] = abs(df['call_delta']) / abs(df['put_delta'])

# 4. Save
df.dropna(inplace=True)
df.to_csv("../data/nifty_features_5min.csv")
print("Features Engineered & Saved.")

 Starting Feature Engineering...
Features Engineered & Saved.
